This notebook demonstrates how structured data stored in an SQL database can be connected with a Large Language Model (LLM) to build a retrieval-augmented generation workflow. Instead of manually writing SQL queries for every user question, the workflow uses an LLM-based query builder to interpret the natural language question and generate an appropriate SQL query.

In [2]:
#install packages
!pip install langchain_openai -q
!pip install langchain_community -q
!pip install langchain -q

In [3]:
#Create SQLite database and tables.
import sqlite3

In [ ]:
#create a connection with a table. table will be created newly in the same location the notebook resides. 
connection = sqlite3.connect("school.db")

In [5]:
#create a cursor to run the queries on the connection.
cursor = connection.cursor()

In [ ]:
#run queries to the data base. first we create three tabels with the database. 
cursor.execute("""
CREATE TABLE IF NOT EXISTS teachers (
    teacher_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    age INTEGER NOT NULL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS subjects (
    subject_id INTEGER PRIMARY KEY AUTOINCREMENT,
    subject_name TEXT NOT NULL
)
""")
#a tabel for the work of teaching. two columsn for subject and the corresponding teacher. 
cursor.execute("""
CREATE TABLE IF NOT EXISTS teaches (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    teacher_id INTEGER,
    subject_id INTEGER,
    FOREIGN KEY (teacher_id) REFERENCES teachers(teacher_id),
    FOREIGN KEY (subject_id) REFERENCES subjects(subject_id)
)
""")

connection.commit()

In [7]:
#insert sample data into tables
teachers = [
    ("Kamal", 42),
    ("Saman", 29),
    ("Pawan", 34)
]
cursor.executemany("INSERT INTO teachers (name, age) VALUES (?, ?)", teachers)

subjects = [
    ("Mathematics",),
    ("Science",),
    ("History",),
    ("English",)
]
cursor.executemany("INSERT INTO subjects (subject_name) VALUES (?)", subjects)

teaches = [
    (1, 1),  # Kamal teaches Mathematics
    (1, 2),  # Kamal teaches Science
    (2, 3),  # Saman teaches History
    (3, 4),  # Pawan teaches English
]
cursor.executemany("INSERT INTO teaches (teacher_id, subject_id) VALUES (?, ?)", teaches)

connection.commit()

In [8]:
#check whether data is available in the tables
cursor.execute("SELECT * FROM teachers")
teachers = cursor.fetchall()
print(teachers)

[(1, 'Kamal', 42), (2, 'Saman', 29), (3, 'Pawan', 34)]


In [ ]:
#Join three tabels based on ids and find the teach who teaches the mathematics 
cursor.execute("""SELECT t.name
FROM teachers t
JOIN teaches te ON t.teacher_id = te.teacher_id
JOIN subjects s ON te.subject_id = s.subject_id
WHERE s.subject_name = 'Mathematics';""")
teachers = cursor.fetchall()
print(teachers)

[('Kamal',)]


Intialize Langchain SQL database

In [11]:
#call the API key as an environment variable
#to manage API key as a local enviornment variable we need OS library, and to load the env variables from .env file we need to install python-dotenv package
%pip install python-dotenv

import os

#load openai key from .env file, first import the library to load env variables
from dotenv import load_dotenv
from pathlib import Path

env_path=Path.cwd() /'.env'#give the .env file path
print("Path to .env file:", env_path)
print("File exists:", env_path.exists())
# Load environment variables from .env file
load_dotenv(env_path,override=True)


#OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
api_key = os.getenv("OPENAI_API_KEY")
print("Loaded:", api_key is not None)

Note: you may need to restart the kernel to use updated packages.
Path to .env file: c:\Users\shara\OneDrive\Documents\Coding Stuff\Generative AI\LangChain\.env
File exists: True
Loaded: True


In [12]:
from langchain_community.utilities.sql_database import SQLDatabase

C:\Users\shara\AppData\Local\Temp\ipykernel_3232\2983476473.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities.sql_database import SQLDatabase


In [13]:
#link our created database to langchain sqldatabase
db = SQLDatabase.from_uri(f"sqlite:///school.db")

In [14]:
#check the db 
print(db.get_usable_table_names())

['subjects', 'teachers', 'teaches']


In [15]:
print(db.table_info)


CREATE TABLE subjects (
	subject_id INTEGER, 
	subject_name TEXT NOT NULL, 
	PRIMARY KEY (subject_id)
)

/*
3 rows from subjects table:
subject_id	subject_name
1	Mathematics
2	Science
3	History
*/


CREATE TABLE teachers (
	teacher_id INTEGER, 
	name TEXT NOT NULL, 
	age INTEGER NOT NULL, 
	PRIMARY KEY (teacher_id)
)

/*
3 rows from teachers table:
teacher_id	name	age
1	Kamal	42
2	Saman	29
3	Pawan	34
*/


CREATE TABLE teaches (
	id INTEGER, 
	teacher_id INTEGER, 
	subject_id INTEGER, 
	PRIMARY KEY (id), 
	FOREIGN KEY(teacher_id) REFERENCES teachers (teacher_id), 
	FOREIGN KEY(subject_id) REFERENCES subjects (subject_id)
)

/*
3 rows from teaches table:
id	teacher_id	subject_id
1	1	1
2	1	2
3	2	3
*/


Intialize the LLM withopenAI

In [16]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

c:\Users\shara\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Create the SQL query generator using Langchain database and LLM 

In [18]:
from langchain_classic.chains.sql_database.query import create_sql_query_chain

query_generate = create_sql_query_chain(llm, db)

Create the sql exectution tool 

In [19]:
from langchain_community.tools import QuerySQLDatabaseTool

query_execute = QuerySQLDatabaseTool(db=db)

In [ ]:
#run a query on the database with langchain query geenrator and the executor.
# first the sql query generator conver the general question into query.
query = query_generate.invoke({"question": "Details of all teachers"})
print(query)

SELECT "teacher_id", "name", "age" FROM teachers


In [21]:
#then the executor run that query
query_execute.invoke(query)

"[(1, 'Kamal', 42), (2, 'Saman', 29), (3, 'Pawan', 34)]"

In [22]:
query = query_generate.invoke({"question": "Which teachers are assigned to teach Mathematics?"})
print(query)

SELECT "name"
FROM teachers
JOIN teaches ON teachers.teacher_id = teaches.teacher_id
JOIN subjects ON teaches.subject_id = subjects.subject_id
WHERE subjects.subject_name = 'Mathematics'
LIMIT 5;


In [23]:
query_execute.invoke(query)

"[('Kamal',)]"

Create Answer Generator Chain with this SQL setup

In [ ]:
#frist create the answer generaton chain using the prompt with the question, sql query and the result
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

answer_prompt = PromptTemplate.from_template(
    """Given a user question, the generated SQL query, and its result, write a clear and natural answer to the question.

    User Question: {question}
    SQL Query: {query}
    SQL Result: {result}

    Final Answer:"""
)
#finally the answer generation chain
final_answer = answer_prompt | llm | StrOutputParser()

In [25]:
#create the coverall chain by combining the executing the query part and the final answer
chain = (
    RunnablePassthrough.assign(query=query_generate).assign(
        result=itemgetter("query") | query_execute
    )
    | final_answer
)

In [26]:
#check the chain
chain.invoke({"question": "Details of all teachers"})

'Here are the details of all teachers: \n1. Teacher ID: 1, Name: Kamal, Age: 42\n2. Teacher ID: 2, Name: Saman, Age: 29\n3. Teacher ID: 3, Name: Pawan, Age: 34'

In [27]:
chain.invoke({"question": "Which teachers are assigned to teach Mathematics?"})

'Kamal is the teacher assigned to teach Mathematics.'